# 66 · Post-E50 — Multiseries DICOM Geometry Pairing

**Objetivo visible:** construir y validar una representación espacial común para series
Sagittal T1, Sagittal T2 y Axial T2 de un mismo estudio lumbar, permitiendo proyectar
puntos/ROIs entre series y seleccionar cortes axiales físicamente correspondientes.

## Problema a resolver

> ¿Cómo relacionar físicamente Sagittal T1, Sagittal T2 y Axial T2 pertenecientes al mismo
> estudio lumbar utilizando la geometría DICOM?

Este notebook **NO** detecta patologías, **NO** entrena redes neuronales y **NO** implementa
naming automático de niveles lumbares (L1-L2 … L5-S1). Esa validación corresponde al futuro
`67_postE50_level_localization_v2.ipynb`.

## Alcance de este notebook

**SÍ:**
- clasifica/identifica las series necesarias por heurística documentada;
- analiza geometría DICOM real (`ImageOrientationPatient`, `ImagePositionPatient`, `PixelSpacing`, etc.);
- construye sistemas de coordenadas y ordena slices físicamente;
- proyecta pixel → patient XYZ y patient XYZ → pixel/plane;
- calcula distancia punto-plano y encuentra slices axiales candidatos;
- consume una referencia anatómica manual (no automática) para validar la geometría;
- construye un `LevelSeriesBundle` **experimental** cuando existe una referencia con nivel conocido.

**NO:**
- crea o entrena un modelo para detectar vértebras/discos;
- afirma que sabe automáticamente qué disco es L4-L5 sin una fuente confiable para ese label;
- cambia `AUTOMATIC_DISC_LOCALIZATION_VALIDATED` a `True` (no se toca código productivo).

## Decisión de diseño: sin paquete `lib/` externo

Todas las funciones de geometría se definen **inline**, dentro de este mismo notebook, en vez de
crear `notebooks/post_e50/lib/` o `research/post_e50/`. Justificación: el notebook es la unidad
de auditoría de esta fase; mantener las funciones inline evita problemas de `sys.path`, mantiene
el experimento autocontenido en un solo archivo versionado, y no crea código "productivo" fuera
de `ai_service/` que después haya que reconciliar. Si estas funciones deben reutilizarse en
producción, su promoción a un módulo de `ai_service/` es trabajo de un ticket posterior explícito.

Rama: `research/post-e50-series-pairing` · Rama madre: `research/post-e50-ai-vnext`


In [1]:
# --- Setup: repository root, allowed write scope, git identity, privacy helpers ---
import hashlib
import io
import json
import os
import subprocess
import sys
import zipfile
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom


def _find_repo_root(start: Path) -> Path:
    # Walk upward from the notebook location until we find repo markers.
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "ai_service").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook location")


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = _find_repo_root(NOTEBOOK_DIR)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
PAIRING_DIR = REPO_ROOT / "artifacts" / "post_e50" / "series_pairing"
FIGURES_DIR = PAIRING_DIR / "figures"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"

warnings: list[str] = []
limitations: list[str] = []

# Terms that must never appear in anything persisted to disk (section 6 / 28).
FORBIDDEN_IDENTIFIER_FIELDS = (
    "PatientName", "PatientID", "AccessionNumber",
    "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "InstitutionName",
)


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS):
        raise RuntimeError(f"Refusing to write outside allowed Post-E50 trees: {path}")
    forbidden_values = getattr(safe_write_text, "_forbidden_values", set())
    for value in forbidden_values:
        if value and value in content:
            raise RuntimeError(f"Refusing to persist forbidden identifier value into {path.name}")
    if "C:\\Users\\" in content or "/Users/" in content:
        raise RuntimeError(f"Refusing to persist a local filesystem path into {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def opaque_id(raw_uid: str) -> str:
    return hashlib.sha256(str(raw_uid).encode("utf-8")).hexdigest()[:12]


def run_git(*args: str) -> str:
    result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
    return result.stdout.strip()


GIT_BRANCH = run_git("branch", "--show-current")
GIT_COMMIT = run_git("rev-parse", "HEAD")
GIT_STATUS_PORCELAIN = run_git("status", "--porcelain")
GENERATED_AT = datetime.now(timezone.utc).isoformat()

print("REPO_ROOT (relative label only, no absolute path persisted):", REPO_ROOT.name)
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)
print("GENERATED_AT:", GENERATED_AT)
print("Working tree clean:", GIT_STATUS_PORCELAIN == "")


REPO_ROOT (relative label only, no absolute path persisted): post-e50-baseline-inventory-cce171
GIT_BRANCH: research/post-e50-series-pairing
GIT_COMMIT: 835373adf763e02be144e7086653a0a173f71879
GENERATED_AT: 2026-08-17T23:23:00.196908+00:00
Working tree clean: False


## 1. Fuente de datos DICOM (read-only)

Se busca un estudio DICOM lumbar de-identificado en ubicaciones **hermanas** del repositorio,
sin copiarlo ni moverlo. Resolución en orden de prioridad:

1. Variable de entorno `PFI_POST_E50_DICOM_STUDY` (ruta a ZIP o directorio DICOM).
2. Búsqueda dinámica ascendente desde `REPO_ROOT` hasta encontrar una carpeta ancestro llamada
   `PFI_MVP_Juntos`, y dentro de ella las carpetas hermanas `PFI_RM_Lumbar_Final` /
   `PFI_RM_Lumbar_Profesores`, buscando `test-data/*.zip`.

Ninguna ruta absoluta local se hardcodea en el notebook ni se persiste en outputs: solo se
imprimen/guardan rutas **relativas** al ancestro común detectado en tiempo de ejecución.


In [2]:
def _redact_relative(path: Path, anchor_name: str = "PFI_MVP_Juntos") -> str:
    # Never expose the filename itself: sibling DICOM export filenames commonly embed the raw
    # study/patient identifier (e.g. "<studyId>-....zip"), which would leak it verbatim into
    # persisted notebook output even though this is a path, not an extracted DICOM field.
    parts = path.resolve().parts
    masked_name = f"<{opaque_id(path.name)}>{path.suffix}"
    if anchor_name in parts:
        idx = parts.index(anchor_name)
        dir_parts = list(parts[idx:-1])
        return "/".join([*dir_parts, masked_name])
    return masked_name


def discover_dicom_source() -> tuple[Path | None, str]:
    env_value = os.environ.get("PFI_POST_E50_DICOM_STUDY")
    if env_value:
        candidate = Path(env_value)
        if candidate.exists():
            return candidate, "env:PFI_POST_E50_DICOM_STUDY"
        warnings.append("PFI_POST_E50_DICOM_STUDY is set but does not exist on disk.")

    anchor = None
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if parent.name == "PFI_MVP_Juntos":
            anchor = parent
            break
    if anchor is None:
        return None, "not_found:no_PFI_MVP_Juntos_ancestor"

    sibling_names = ["PFI_RM_Lumbar_Final", "PFI_RM_Lumbar_Profesores"]
    candidates = []
    for name in sibling_names:
        sibling = anchor / name
        test_data = sibling / "test-data"
        if test_data.is_dir():
            candidates.extend(sorted(test_data.glob("*.zip")))

    if not candidates:
        return None, "not_found:no_test_data_zip_in_siblings"

    # Prefer a zip whose SHA-256 matches the historically expected study hash.
    expected_sha = "1C058033AADAF9C72AF8F1B5D85DBBBDCB7706FDDA50B74537CE71A55B2227B1".lower()
    for cand in candidates:
        digest = hashlib.sha256()
        with cand.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() == expected_sha:
            return cand, f"sibling_search:{_redact_relative(cand)}"

    return candidates[0], f"sibling_search_no_hash_match:{_redact_relative(candidates[0])}"


DICOM_SOURCE_PATH, DICOM_SOURCE_METHOD = discover_dicom_source()
REAL_DICOM_USED = DICOM_SOURCE_PATH is not None

print("DICOM source found:", REAL_DICOM_USED)
print("Discovery method:", DICOM_SOURCE_METHOD)
if DICOM_SOURCE_PATH is not None:
    print("Relative label:", _redact_relative(DICOM_SOURCE_PATH))


DICOM source found: True
Discovery method: sibling_search:PFI_MVP_Juntos/PFI_RM_Lumbar_Final/test-data/<c40f992f99ed>.zip
Relative label: PFI_MVP_Juntos/PFI_RM_Lumbar_Final/test-data/<c40f992f99ed>.zip


In [3]:
EXPECTED_STUDY_ZIP_SHA256 = "1C058033AADAF9C72AF8F1B5D85DBBBDCB7706FDDA50B74537CE71A55B2227B1".lower()
dicom_zip_sha256 = None
dicom_source_size_bytes = None

if REAL_DICOM_USED and DICOM_SOURCE_PATH.is_file():
    digest = hashlib.sha256()
    with DICOM_SOURCE_PATH.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    dicom_zip_sha256 = digest.hexdigest()
    dicom_source_size_bytes = DICOM_SOURCE_PATH.stat().st_size
    hash_matches_expected = dicom_zip_sha256 == EXPECTED_STUDY_ZIP_SHA256
    print("SHA-256:", dicom_zip_sha256)
    print("Matches historically expected study hash:", hash_matches_expected)
    if not hash_matches_expected:
        warnings.append(
            "Located DICOM zip SHA-256 does not match the historically expected value. "
            "Treated as a different dataset/artifact, not as corruption."
        )
elif not REAL_DICOM_USED:
    warnings.append("No real DICOM study located; PFI_POST_E50_DICOM_STUDY not set and sibling search failed.")

print("Size bytes:", dicom_source_size_bytes)


SHA-256: 1c058033aadaf9c72af8f1b5d85dbbbdcb7706fdda50b74537ce71a55b2227b1
Matches historically expected study hash: True
Size bytes: 11523826


## 2. Lectura de instancias DICOM directamente desde el ZIP (sin extraer al repo)

Se lee cada `.dcm` en memoria (`zipfile` + `BytesIO`) usando `pydicom`. Nunca se escribe un
archivo DICOM ni un array de píxeles dentro del repositorio.


In [4]:
@dataclass
class DicomInstance:
    series_folder: str
    entry_name: str
    dataset: "pydicom.dataset.FileDataset"


def _iter_dicom_entries(source: Path):
    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            for name in zf.namelist():
                if name.endswith(".dcm"):
                    with zf.open(name) as handle:
                        data = handle.read()
                    parts = name.split("/")
                    series_folder = parts[1] if len(parts) > 1 else "unknown"
                    yield series_folder, name, data
    elif source.is_dir():
        for path in source.rglob("*.dcm"):
            series_folder = path.parent.name
            yield series_folder, str(path.relative_to(source)), path.read_bytes()
    else:
        return


def load_instances_metadata_only(source: Path) -> list[DicomInstance]:
    instances = []
    for series_folder, entry_name, data in _iter_dicom_entries(source):
        ds = pydicom.dcmread(io.BytesIO(data), stop_before_pixels=True)
        instances.append(DicomInstance(series_folder, entry_name, ds))
    return instances


def load_one_pixel_array(source: Path, entry_name: str) -> np.ndarray:
    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            with zf.open(entry_name) as handle:
                data = handle.read()
        ds = pydicom.dcmread(io.BytesIO(data))
    else:
        ds = pydicom.dcmread(source / entry_name)
    return ds.pixel_array.astype(np.float32)


if REAL_DICOM_USED:
    instances = load_instances_metadata_only(DICOM_SOURCE_PATH)
    print("Total DICOM instances found:", len(instances))
    series_folders_found = sorted(set(inst.series_folder for inst in instances))
    print("Series folders:", series_folders_found)
else:
    instances = []
    print("No instances loaded (no real DICOM source).")


Total DICOM instances found: 48
Series folders: ['2539455828', '2720025375', '3775545364']


## 2b. Frame of Reference — corrección crítica de cross-series geometry

**No se asume** que dos series con `FrameOfReferenceUID` diferente comparten automáticamente un
espacio de coordenadas paciente directamente comparable. Las transformaciones pixel ↔ patient
siguen siendo válidas **dentro** del `FrameOfReference` de cada serie individual; lo que cambia
es cómo se interpreta una distancia calculada **entre** series con FoR distinto.

Se busca, de forma read-only, evidencia DICOM explícita de un objeto de Spatial/Deformable
Registration (`SOPClassUID` de Spatial Registration Storage `1.2.840.10008.5.1.4.1.1.66.1` o
Deformable Spatial Registration Storage `1.2.840.10008.5.1.4.1.1.66.3`, o presencia de
`RegistrationSequence`/`DeformableRegistrationSequence`) entre las instancias ya cargadas del
estudio. **No se inventa ninguna transformación** — si no se encuentra evidencia, se marca
explícitamente que no hay registro disponible.

Clasificación resultante para cada par de series (`classify_frame_relationship`):

- **A. `DIRECT_DICOM_GEOMETRY_ALLOWED`** — mismo `FrameOfReferenceUID`.
- **B. `REGISTERED_DICOM_GEOMETRY_ALLOWED`** — FoR distinto, pero existe una transformación
  DICOM explícita encontrada entre ambos frames.
- **C. `DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED`** — FoR distinto y no se encontró
  transformación explícita. Este es el caso por defecto cuando no hay evidencia.


In [5]:
REGISTRATION_SOP_CLASS_UIDS = {
    "1.2.840.10008.5.1.4.1.1.66.1": "spatial_registration_storage",
    "1.2.840.10008.5.1.4.1.1.66.3": "deformable_spatial_registration_storage",
}

registration_objects_found = []
if REAL_DICOM_USED:
    for inst in instances:
        ds = inst.dataset
        sop_class = str(getattr(ds, "SOPClassUID", ""))
        has_reg_seq = hasattr(ds, "RegistrationSequence") or hasattr(ds, "DeformableRegistrationSequence")
        if sop_class in REGISTRATION_SOP_CLASS_UIDS or has_reg_seq:
            registration_objects_found.append({
                "series_folder": inst.series_folder,
                "sop_class_kind": REGISTRATION_SOP_CLASS_UIDS.get(sop_class, "registration_sequence_tag_present"),
            })

# Populated ONLY from explicit DICOM registration evidence found above; never fabricated.
# Keyed by (for_uid_opaque_a, for_uid_opaque_b) -> transform metadata. Empty unless a real
# Spatial/Deformable Registration object is found in the source study.
FRAME_OF_REFERENCE_TRANSFORMS: dict[tuple[str, str], dict] = {}

print("Registration objects found (read-only search over all loaded instances):", len(registration_objects_found))
for r in registration_objects_found:
    print(" -", r)
if not registration_objects_found:
    print("No explicit DICOM Spatial/Deformable Registration object found in this study.")
    print("Any cross-series comparison with differing FrameOfReferenceUID will be classified as")
    print("DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED (case C), not assumed compatible.")


def classify_frame_relationship(for_uid_opaque_a, for_uid_opaque_b) -> str:
    if for_uid_opaque_a is not None and for_uid_opaque_a == for_uid_opaque_b:
        return "DIRECT_DICOM_GEOMETRY_ALLOWED"
    if (for_uid_opaque_a, for_uid_opaque_b) in FRAME_OF_REFERENCE_TRANSFORMS or (for_uid_opaque_b, for_uid_opaque_a) in FRAME_OF_REFERENCE_TRANSFORMS:
        return "REGISTERED_DICOM_GEOMETRY_ALLOWED"
    return "DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED"


Registration objects found (read-only search over all loaded instances): 0
No explicit DICOM Spatial/Deformable Registration object found in this study.
Any cross-series comparison with differing FrameOfReferenceUID will be classified as
DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED (case C), not assumed compatible.


## 3. Geometría DICOM: transformaciones pixel ↔ patient

**Convención verificada (pydicom / DICOM PS3.3 C.7.6.2.1.1):**

`ImageOrientationPatient = [r1, r2, r3, c1, c2, c3]` donde:

- `row_cosines = (r1, r2, r3)` — dirección en la que se mueve el punto físico cuando el
  **índice de columna** aumenta (recorrer una fila de izquierda a derecha).
- `column_cosines = (c1, c2, c3)` — dirección en la que se mueve el punto físico cuando el
  **índice de fila** aumenta (recorrer una columna de arriba a abajo).

Fórmula oficial pixel → patient:

```
patient = ImagePositionPatient
        + col_index * PixelSpacing[1] * row_cosines
        + row_index * PixelSpacing[0] * column_cosines
```

`PixelSpacing = [spacing_between_rows, spacing_between_columns]` (orden DICOM estándar).

Esta nomenclatura (`row_cosines` para los primeros 3 valores, `column_cosines` para los
últimos 3) es exactamente la que usa `pydicom` en su propia documentación, elegida
deliberadamente para no confundir "row direction" con "el eje que cambia cuando avanza la
fila" (que en realidad es `column_cosines`, no `row_cosines`) — la trampa que advierte el
brief de esta fase.

`normal = cross(row_cosines, column_cosines)` (convención right-handed estándar).


In [6]:
def row_column_cosines(image_orientation_patient) -> tuple[np.ndarray, np.ndarray]:
    iop = np.asarray(image_orientation_patient, dtype=np.float64)
    return iop[:3], iop[3:]


def plane_normal(row_cosines: np.ndarray, column_cosines: np.ndarray) -> np.ndarray:
    return np.cross(row_cosines, column_cosines)


def pixel_to_patient_xyz(image_position, image_orientation, pixel_spacing, row, col) -> np.ndarray:
    # Map a (row, col) pixel index to patient-space XYZ in mm.
    ipp = np.asarray(image_position, dtype=np.float64)
    row_cos, col_cos = row_column_cosines(image_orientation)
    d_row, d_col = float(pixel_spacing[0]), float(pixel_spacing[1])
    return ipp + col * d_col * row_cos + row * d_row * col_cos


def patient_xyz_to_slice_coordinates(point_xyz, image_position, image_orientation, pixel_spacing):
    # Inverse map: patient XYZ -> (row_float, col_float) within the plane of this slice.
    # Assumes the point lies in (or is projected onto) the plane; the in-plane projection is
    # obtained via orthogonal decomposition against the orthonormal row/column cosines.
    ipp = np.asarray(image_position, dtype=np.float64)
    row_cos, col_cos = row_column_cosines(image_orientation)
    d_row, d_col = float(pixel_spacing[0]), float(pixel_spacing[1])
    diff = np.asarray(point_xyz, dtype=np.float64) - ipp
    col = float(np.dot(diff, row_cos)) / d_col
    row = float(np.dot(diff, col_cos)) / d_row
    return row, col


@dataclass
class Plane:
    origin_xyz: np.ndarray
    row_direction: np.ndarray
    column_direction: np.ndarray
    normal: np.ndarray
    rows: int
    columns: int
    pixel_spacing: tuple[float, float]


def build_plane(image_position, image_orientation, pixel_spacing, rows, columns) -> Plane:
    row_cos, col_cos = row_column_cosines(image_orientation)
    normal = plane_normal(row_cos, col_cos)
    return Plane(
        origin_xyz=np.asarray(image_position, dtype=np.float64),
        row_direction=row_cos,
        column_direction=col_cos,
        normal=normal / np.linalg.norm(normal),
        rows=int(rows),
        columns=int(columns),
        pixel_spacing=(float(pixel_spacing[0]), float(pixel_spacing[1])),
    )


def signed_distance_point_to_plane(point_xyz, plane: Plane) -> float:
    return float(np.dot(np.asarray(point_xyz, dtype=np.float64) - plane.origin_xyz, plane.normal))


def absolute_distance_point_to_plane(point_xyz, plane: Plane) -> float:
    return abs(signed_distance_point_to_plane(point_xyz, plane))


def project_patient_point_to_image_plane(point_xyz, plane: Plane) -> dict:
    row, col = patient_xyz_to_slice_coordinates(
        point_xyz, plane.origin_xyz, np.concatenate([plane.row_direction, plane.column_direction]), plane.pixel_spacing
    )
    distance_mm = signed_distance_point_to_plane(point_xyz, plane)
    inside_fov = (-0.5 <= row <= plane.rows - 0.5) and (-0.5 <= col <= plane.columns - 0.5)
    return {
        "row_float": row,
        "col_float": col,
        "inside_fov": bool(inside_fov),
        "distance_to_plane_mm": distance_mm,
    }


print("Geometry primitives defined: pixel_to_patient_xyz, patient_xyz_to_slice_coordinates,")
print("build_plane, signed_distance_point_to_plane, absolute_distance_point_to_plane,")
print("project_patient_point_to_image_plane")


Geometry primitives defined: pixel_to_patient_xyz, patient_xyz_to_slice_coordinates,
build_plane, signed_distance_point_to_plane, absolute_distance_point_to_plane,
project_patient_point_to_image_plane


## 4. Tests geométricos deterministicos (obligatorios antes de tocar datos reales)

Tests A–F según el brief. Si cualquiera falla, `geometry_unit_tests = FAIL` y el quality gate
correspondiente (GATE B / GATE C) se marca `FAIL` — no se avanza asumiendo que la matemática es
correcta.


In [7]:
geometry_test_results = {}

# Deterministic synthetic geometry (geometric_test_reference): axis-aligned sagittal-like plane.
TEST_IPP = np.array([10.0, -20.0, 30.0])
TEST_IOP = [0.0, 1.0, 0.0, 0.0, 0.0, -1.0]  # row_cosines=+Y, column_cosines=-Z
TEST_SPACING = (0.5, 0.8)  # (d_row, d_col)
TEST_ROWS, TEST_COLS = 64, 64

# Test A: pixel (0,0) must map exactly to ImagePositionPatient.
p00 = pixel_to_patient_xyz(TEST_IPP, TEST_IOP, TEST_SPACING, row=0, col=0)
geometry_test_results["A_pixel_00_equals_ipp"] = bool(np.allclose(p00, TEST_IPP, atol=1e-9))

# Test B: incrementing row by 1 must shift XYZ by exactly d_row * column_cosines.
p10 = pixel_to_patient_xyz(TEST_IPP, TEST_IOP, TEST_SPACING, row=1, col=0)
expected_row_delta = TEST_SPACING[0] * np.array(TEST_IOP[3:])
geometry_test_results["B_row_increment_matches_spacing"] = bool(np.allclose(p10 - p00, expected_row_delta, atol=1e-9))

# Test C: incrementing column by 1 must shift XYZ by exactly d_col * row_cosines.
p01 = pixel_to_patient_xyz(TEST_IPP, TEST_IOP, TEST_SPACING, row=0, col=1)
expected_col_delta = TEST_SPACING[1] * np.array(TEST_IOP[:3])
geometry_test_results["C_col_increment_matches_spacing"] = bool(np.allclose(p01 - p00, expected_col_delta, atol=1e-9))

# Test D: plane normal must have norm ~= 1.
row_cos, col_cos = row_column_cosines(TEST_IOP)
normal = plane_normal(row_cos, col_cos)
geometry_test_results["D_normal_unit_norm"] = bool(abs(np.linalg.norm(normal) - 1.0) < 1e-9)

# Test E: row and column directions must be approximately orthogonal.
geometry_test_results["E_row_col_orthogonal"] = bool(abs(np.dot(row_cos, col_cos)) < 1e-9)

# Test F: round trip pixel -> patient -> pixel, error under tolerance (<=1e-3 pixel).
rng = np.random.default_rng(2026)
roundtrip_errors = []
for _ in range(200):
    r = rng.uniform(0, TEST_ROWS - 1)
    c = rng.uniform(0, TEST_COLS - 1)
    xyz = pixel_to_patient_xyz(TEST_IPP, TEST_IOP, TEST_SPACING, row=r, col=c)
    r2, c2 = patient_xyz_to_slice_coordinates(xyz, TEST_IPP, TEST_IOP, TEST_SPACING)
    roundtrip_errors.append(max(abs(r2 - r), abs(c2 - c)))
max_roundtrip_error = float(max(roundtrip_errors))
geometry_test_results["F_roundtrip_error_under_tolerance"] = bool(max_roundtrip_error <= 1e-3)

GEOMETRY_UNIT_TESTS_PASS = all(geometry_test_results.values())
print(pd.Series(geometry_test_results))
print()
print("max_roundtrip_error:", max_roundtrip_error)
print("geometry_unit_tests:", "PASS" if GEOMETRY_UNIT_TESTS_PASS else "FAIL")
if not GEOMETRY_UNIT_TESTS_PASS:
    warnings.append("Deterministic geometry unit tests FAILED; downstream geometry results are not trustworthy.")


A_pixel_00_equals_ipp                True
B_row_increment_matches_spacing      True
C_col_increment_matches_spacing      True
D_normal_unit_norm                   True
E_row_col_orthogonal                 True
F_roundtrip_error_under_tolerance    True
dtype: bool

max_roundtrip_error: 7.105427357601002e-15
geometry_unit_tests: PASS


## 5. Series discovery y clasificación heurística de rol

Heurística documentada (no afirma certeza clínica):

1. Se calcula el **eje dominante de la normal del plano** a partir de `ImageOrientationPatient`
   de una instancia representativa de la serie:
   - dominante en **X** (eje left-right del paciente) → candidato `sagittal`;
   - dominante en **Z** (eje superior-inferior) → candidato `axial` (incluye axiales oblicuos,
     comunes en protocolos lumbares alineados por disco);
   - dominante en **Y** → candidato `coronal`, mapeado aquí a `other`.
2. Dentro de cada candidato geométrico, se usa `SeriesDescription` (saneada) para distinguir
   T1 de T2 cuando el texto contiene esas subcadenas.
3. Si falta `ImageOrientationPatient` o hay instancias inconsistentes, el rol es `unknown`.

`role_confidence` combina ambas señales; se documenta la fórmula exacta en el código.


In [8]:
def sanitize_series_description(value) -> str:
    if value is None:
        return ""
    text = str(value)
    # SeriesDescription here only carries short sequence labels (e.g. "T1"/"T2"); still cap
    # length defensively in case an unexpected free-text value appears.
    return text[:64]


def classify_series_role(sample_ds, description_sanitized: str) -> tuple[str, float, list[str]]:
    reasons = []
    if not hasattr(sample_ds, "ImageOrientationPatient") or sample_ds.ImageOrientationPatient is None:
        return "unknown", 0.0, ["missing_image_orientation_patient"]

    row_cos, col_cos = row_column_cosines(sample_ds.ImageOrientationPatient)
    normal = plane_normal(row_cos, col_cos)
    dominant_axis = int(np.argmax(np.abs(normal)))
    axis_names = ("x", "y", "z")
    reasons.append(f"normal_dominant_axis={axis_names[dominant_axis]} ({normal.round(4).tolist()})")

    orientation_geometry_score = 0.0
    geometric_family = "other"
    if dominant_axis == 0:
        geometric_family = "sagittal"
        orientation_geometry_score = 0.5
    elif dominant_axis == 2:
        geometric_family = "axial"
        orientation_geometry_score = 0.5
    else:
        geometric_family = "other"
        orientation_geometry_score = 0.1

    desc_lower = description_sanitized.lower()
    text_score = 0.0
    weighting = "t1" if "t1" in desc_lower else ("t2" if "t2" in desc_lower else None)
    if weighting:
        text_score = 0.3
        reasons.append(f"series_description_contains={weighting}")
    else:
        reasons.append("series_description_did_not_disambiguate_weighting")

    if geometric_family == "sagittal" and weighting == "t1":
        role = "sagittal_t1"
    elif geometric_family == "sagittal" and weighting == "t2":
        role = "sagittal_t2"
    elif geometric_family == "axial" and weighting == "t2":
        role = "axial_t2"
    elif geometric_family in ("sagittal", "axial"):
        role = "other"
        reasons.append("geometric_family_known_but_weighting_undetermined")
    else:
        role = "other"

    confidence = min(1.0, orientation_geometry_score + text_score)
    return role, float(confidence), reasons


print("classify_series_role defined (heuristic; documented, not a clinical claim).")


classify_series_role defined (heuristic; documented, not a clinical claim).

In [9]:
def slice_thickness_or_none(ds):
    return float(ds.SliceThickness) if hasattr(ds, "SliceThickness") and ds.SliceThickness is not None else None


def spacing_between_slices_or_none(ds):
    return float(ds.SpacingBetweenSlices) if hasattr(ds, "SpacingBetweenSlices") and ds.SpacingBetweenSlices is not None else None


series_rows = []
series_geometry: dict[str, dict] = {}

if REAL_DICOM_USED:
    study_uid_raw = None
    by_series: dict[str, list[DicomInstance]] = {}
    for inst in instances:
        sid = str(inst.dataset.SeriesInstanceUID)
        by_series.setdefault(sid, []).append(inst)
        if study_uid_raw is None and hasattr(inst.dataset, "StudyInstanceUID"):
            study_uid_raw = str(inst.dataset.StudyInstanceUID)

    STUDY_OPAQUE_ID = opaque_id(study_uid_raw) if study_uid_raw else "unknown_study"
    for series_uid, insts in by_series.items():
        series_opaque = opaque_id(series_uid)
        sample = insts[0].dataset
        description_sanitized = sanitize_series_description(getattr(sample, "SeriesDescription", None))
        role, confidence, reasons = classify_series_role(sample, description_sanitized)

        has_geometry = all(
            hasattr(sample, attr) for attr in ("ImageOrientationPatient", "ImagePositionPatient", "PixelSpacing", "Rows", "Columns")
        )
        row_warnings = []
        positions_scalar = []
        normal = None
        if has_geometry:
            row_cos, col_cos = row_column_cosines(sample.ImageOrientationPatient)
            normal = plane_normal(row_cos, col_cos)
            for inst in insts:
                if not hasattr(inst.dataset, "ImagePositionPatient"):
                    row_warnings.append("instance_missing_image_position_patient")
                    continue
                ipp = np.asarray(inst.dataset.ImagePositionPatient, dtype=np.float64)
                positions_scalar.append(float(np.dot(ipp, normal)))
        else:
            row_warnings.append("series_missing_required_geometry_fields")

        for_uid = getattr(sample, "FrameOfReferenceUID", None)

        series_rows.append({
            "study_opaque_id": STUDY_OPAQUE_ID,
            "series_opaque_id": series_opaque,
            "series_description_sanitized": description_sanitized,
            "modality": getattr(sample, "Modality", None) or "UNKNOWN_NOT_PRESENT",
            "rows": int(sample.Rows) if hasattr(sample, "Rows") else None,
            "columns": int(sample.Columns) if hasattr(sample, "Columns") else None,
            "slice_count": len(insts),
            "pixel_spacing_row_mm": float(sample.PixelSpacing[0]) if hasattr(sample, "PixelSpacing") else None,
            "pixel_spacing_col_mm": float(sample.PixelSpacing[1]) if hasattr(sample, "PixelSpacing") else None,
            "slice_thickness_mm": slice_thickness_or_none(sample),
            "spacing_between_slices_mm": spacing_between_slices_or_none(sample),
            "orientation_class": ("sagittal" if role.startswith("sagittal") else "axial" if role == "axial_t2" else "other"),
            "plane_normal_x": float(normal[0]) if normal is not None else None,
            "plane_normal_y": float(normal[1]) if normal is not None else None,
            "plane_normal_z": float(normal[2]) if normal is not None else None,
            "position_min": min(positions_scalar) if positions_scalar else None,
            "position_max": max(positions_scalar) if positions_scalar else None,
            "frame_of_reference_match": None,  # filled in after all series are discovered
            "candidate_role": role,
            "role_confidence": confidence,
            "geometry_valid": has_geometry and len(row_warnings) == 0,
            "warnings": "; ".join(reasons + row_warnings),
        })

        series_geometry[series_opaque] = {
            "series_instance_uid_used_internally_only": True,  # never persisted raw
            "for_uid_opaque": opaque_id(for_uid) if for_uid else None,
            "instances": insts,
            "sample_ds": sample,
        }

    # Cross-check FrameOfReferenceUID equality across series (using opaque form only).
    for_values = {sid: g["for_uid_opaque"] for sid, g in series_geometry.items()}
    for row in series_rows:
        sid = row["series_opaque_id"]
        others = [v for k, v in for_values.items() if k != sid]
        row["frame_of_reference_match"] = bool(for_values.get(sid) is not None and for_values[sid] in others)

series_inventory = pd.DataFrame(series_rows)
series_inventory


,study_opaque_id,series_opaque_id,series_description_sanitized,modality,rows,columns,slice_count,pixel_spacing_row_mm,pixel_spacing_col_mm,slice_thickness_mm,...,plane_normal_x,plane_normal_y,plane_normal_z,position_min,position_max,frame_of_reference_match,candidate_role,role_confidence,geometry_valid,warnings
0,206aa67ee4e6,28ea4937243f,T1,UNKNOWN_NOT_PRESENT,512,512,12,0.5859,0.5859,4.5,...,-0.998661,0.049926,-0.013281,-22.710411,37.789333,False,sagittal_t1,0.8,True,"normal_dominant_axis=x ([-0.9987, 0.0499, -0.0..."
1,206aa67ee4e6,b65eca7cfbbc,T2,UNKNOWN_NOT_PRESENT,512,512,24,0.3906,0.3906,4.5,...,0.018891,-0.390318,0.920492,-52.144191,140.216381,False,axial_t2,0.8,True,"normal_dominant_axis=z ([0.0189, -0.3903, 0.92..."
2,206aa67ee4e6,25aa08338722,T2,UNKNOWN_NOT_PRESENT,512,512,12,0.5859,0.5859,4.5,...,-0.998661,0.049926,-0.013281,-22.710411,37.789333,False,sagittal_t2,0.8,True,"normal_dominant_axis=x ([-0.9987, 0.0499, -0.0..."


## 6. Orden físico de slices por serie

Se ordena cada serie por el escalar `s = dot(ImagePositionPatient, normal)`, **no** por
`InstanceNumber`. Se registran estadísticas de spacing observado y se comparan contra
`SliceThickness` / `SpacingBetweenSlices` sin asumir que son equivalentes.


In [10]:
def order_series_physically(series_opaque_id: str) -> dict:
    g = series_geometry[series_opaque_id]
    sample = g["sample_ds"]
    if not hasattr(sample, "ImageOrientationPatient"):
        return {"ordered": [], "median_slice_spacing_mm": None, "min_slice_spacing_mm": None,
                "max_slice_spacing_mm": None, "spacing_std_mm": None, "irregular_spacing": None,
                "duplicates": None, "monotonic": None}

    row_cos, col_cos = row_column_cosines(sample.ImageOrientationPatient)
    normal = plane_normal(row_cos, col_cos)
    scored = []
    for inst in g["instances"]:
        if not hasattr(inst.dataset, "ImagePositionPatient"):
            continue
        ipp = np.asarray(inst.dataset.ImagePositionPatient, dtype=np.float64)
        s = float(np.dot(ipp, normal))
        scored.append((s, inst))
    scored.sort(key=lambda item: item[0])

    scalars = np.array([s for s, _ in scored])
    diffs = np.diff(scalars)
    duplicates = int(np.sum(np.isclose(diffs, 0.0, atol=1e-6)))
    monotonic = bool(np.all(diffs > 0)) or bool(np.all(diffs < 0))

    spacing_stats = {
        "median_slice_spacing_mm": float(np.median(np.abs(diffs))) if len(diffs) else None,
        "min_slice_spacing_mm": float(np.min(np.abs(diffs))) if len(diffs) else None,
        "max_slice_spacing_mm": float(np.max(np.abs(diffs))) if len(diffs) else None,
        "spacing_std_mm": float(np.std(np.abs(diffs))) if len(diffs) else None,
    }
    irregular = None
    if spacing_stats["median_slice_spacing_mm"]:
        ratio = spacing_stats["spacing_std_mm"] / spacing_stats["median_slice_spacing_mm"]
        irregular = bool(ratio > 0.15)

    return {
        "ordered": scored,
        "duplicates": duplicates,
        "monotonic": monotonic,
        "irregular_spacing": irregular,
        **spacing_stats,
    }


slice_ordering = {sid: order_series_physically(sid) for sid in series_geometry} if REAL_DICOM_USED else {}
slice_ordering_summary = pd.DataFrame([
    {
        "series_opaque_id": sid,
        "duplicates": info["duplicates"],
        "monotonic": info["monotonic"],
        "irregular_spacing": info["irregular_spacing"],
        "median_slice_spacing_mm": info["median_slice_spacing_mm"],
        "min_slice_spacing_mm": info["min_slice_spacing_mm"],
        "max_slice_spacing_mm": info["max_slice_spacing_mm"],
        "spacing_std_mm": info["spacing_std_mm"],
    }
    for sid, info in slice_ordering.items()
])
slice_ordering_summary


,series_opaque_id,duplicates,monotonic,irregular_spacing,median_slice_spacing_mm,min_slice_spacing_mm,max_slice_spacing_mm,spacing_std_mm
0,28ea4937243f,0,True,False,5.499975,5.499925,5.500025,0.000037
1,b65eca7cfbbc,0,True,True,5.246698,4.740896,37.889650,8.027674
2,25aa08338722,0,True,False,5.499975,5.499925,5.500025,0.000037


## 6b. Axial orientation clustering — investigación de spacing irregular

El Axial T2 fue marcado con `irregular_spacing=True` en la sección anterior. Antes de
interpretarlo como una única pila mal espaciada, se lee `ImageOrientationPatient` **por cada
slice individual** (no solo una instancia representativa) y se agrupan los slices por similitud
angular de su normal, con una tolerancia técnica explícita — sin asumir de antemano cuántos
clusters existen.

Tolerancia: `ORIENTATION_CLUSTER_ANGLE_TOLERANCE_DEG = 1.0°`, elegida porque los slices con
idéntica orientación dentro de este estudio difieren en `0.0°` entre sí, mientras que el gap
angular mínimo observado entre grupos de orientación distintos es `1.358°` — hay margen claro
para separarlos sin fusionar clusters distintos ni fragmentar uno real.

Agrupamiento por *union-find* sobre todos los pares de slices (no comparación solo contra un
representante fijo), para evitar dependencia del orden de iteración.


In [11]:
ORIENTATION_CLUSTER_ANGLE_TOLERANCE_DEG = 1.0


def angle_between_normals_deg(n1: np.ndarray, n2: np.ndarray) -> float:
    cos_a = float(np.clip(abs(np.dot(n1, n2)) / (np.linalg.norm(n1) * np.linalg.norm(n2)), -1.0, 1.0))
    return float(np.degrees(np.arccos(cos_a)))


def cluster_series_by_orientation(series_opaque_id: str, tolerance_deg: float = ORIENTATION_CLUSTER_ANGLE_TOLERANCE_DEG) -> list[dict]:
    g = series_geometry[series_opaque_id]
    per_slice = []
    for inst in g["instances"]:
        ds = inst.dataset
        if not hasattr(ds, "ImageOrientationPatient") or not hasattr(ds, "ImagePositionPatient"):
            continue
        row_cos, col_cos = row_column_cosines(ds.ImageOrientationPatient)
        normal = plane_normal(row_cos, col_cos)
        normal = normal / np.linalg.norm(normal)
        per_slice.append({"inst": inst, "normal": normal, "ipp": np.asarray(ds.ImagePositionPatient, dtype=np.float64)})

    n = len(per_slice)
    if n == 0:
        return []

    # Union-find over all pairs within tolerance -> robust to iteration order.
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    for i in range(n):
        for j in range(i + 1, n):
            if angle_between_normals_deg(per_slice[i]["normal"], per_slice[j]["normal"]) <= tolerance_deg:
                union(i, j)

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)

    cluster_rows = []
    for cluster_id, (_, indices) in enumerate(sorted(groups.items(), key=lambda kv: kv[0])):
        members = [per_slice[i] for i in indices]
        mean_normal = np.mean([m["normal"] for m in members], axis=0)
        mean_normal = mean_normal / np.linalg.norm(mean_normal)
        max_dev = max(angle_between_normals_deg(m["normal"], mean_normal) for m in members)
        scalars = sorted(float(np.dot(m["ipp"], mean_normal)) for m in members)
        diffs = np.diff(scalars)
        ordering_valid = bool(len(diffs) == 0 or np.all(diffs > 0) or np.all(diffs < 0))
        cluster_rows.append({
            "series_opaque_id": series_opaque_id,
            "cluster_id": cluster_id,
            "slice_count": len(members),
            "mean_normal_x": float(mean_normal[0]),
            "mean_normal_y": float(mean_normal[1]),
            "mean_normal_z": float(mean_normal[2]),
            "max_angular_deviation_deg": float(max_dev),
            "physical_position_min": float(min(scalars)),
            "physical_position_max": float(max(scalars)),
            "median_spacing_mm": float(np.median(np.abs(diffs))) if len(diffs) else None,
            "min_spacing_mm": float(np.min(np.abs(diffs))) if len(diffs) else None,
            "max_spacing_mm": float(np.max(np.abs(diffs))) if len(diffs) else None,
            "spacing_std_mm": float(np.std(np.abs(diffs))) if len(diffs) else None,
            "ordering_valid": ordering_valid,
        })
    return cluster_rows


axial_orientation_cluster_rows = []
if REAL_DICOM_USED:
    axial_ids_for_clustering = series_inventory.loc[series_inventory["candidate_role"] == "axial_t2", "series_opaque_id"].tolist()
    for sid in axial_ids_for_clustering:
        axial_orientation_cluster_rows.extend(cluster_series_by_orientation(sid))

axial_orientation_clusters = pd.DataFrame(axial_orientation_cluster_rows)
AXIAL_ORIENTATION_CLUSTERING_PASS = bool(len(axial_orientation_clusters) > 0 and axial_orientation_clusters["ordering_valid"].all())

axial_orientation_clusters.to_csv(PAIRING_DIR / "post_e50_axial_orientation_clusters.csv", index=False)

print(f"Number of axial orientation clusters found: {len(axial_orientation_clusters)}")
if len(axial_orientation_clusters):
    print(axial_orientation_clusters[["cluster_id", "slice_count", "max_angular_deviation_deg", "ordering_valid"]])
print("AXIAL_ORIENTATION_CLUSTERING_PASS:", AXIAL_ORIENTATION_CLUSTERING_PASS)
if len(axial_orientation_clusters) > 1:
    limitations.append(
        f"Axial T2 series is NOT a single physically-parallel stack: {len(axial_orientation_clusters)} "
        "distinct orientation clusters were found (per-slice ImageOrientationPatient varies), each "
        "ordered independently. This explains the irregular_spacing flag from Section 6, which "
        "was computed treating the whole series as one stack. No cluster is claimed to correspond "
        "to any specific lumbar level -- that is out of scope for Notebook 66."
    )


Number of axial orientation clusters found: 5
   cluster_id  slice_count  max_angular_deviation_deg  ordering_valid
0           0            6                        0.0            True
1           1            4                        0.0            True
2           2            4                        0.0            True
3           3            5                        0.0            True
4           4            5                        0.0            True
AXIAL_ORIENTATION_CLUSTERING_PASS: True


In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
if len(axial_orientation_clusters):
    ax.bar(axial_orientation_clusters["cluster_id"].astype(str), axial_orientation_clusters["slice_count"])
    ax.set_xlabel("orientation cluster_id")
    ax.set_ylabel("slice_count")
    ax.set_title(f"Axial T2 orientation clusters (tolerance={ORIENTATION_CLUSTER_ANGLE_TOLERANCE_DEG} deg)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "axial_orientation_clusters.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_36224\852610584.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Sagittal T1 ↔ Sagittal T2

Sin registration deformable, sin aprendizaje: solo geometría DICOM y una estimación simple de
solapamiento por bounding box axis-aligned en espacio paciente.


In [13]:
def series_corner_points(series_opaque_id: str) -> np.ndarray:
    g = series_geometry[series_opaque_id]
    sample = g["sample_ds"]
    rows, cols = int(sample.Rows), int(sample.Columns)
    corners = []
    for _, inst in slice_ordering[series_opaque_id]["ordered"]:
        ds = inst.dataset
        for r, c in [(0, 0), (0, cols - 1), (rows - 1, 0), (rows - 1, cols - 1)]:
            corners.append(pixel_to_patient_xyz(ds.ImagePositionPatient, ds.ImageOrientationPatient, ds.PixelSpacing, r, c))
    return np.array(corners)


def bbox_overlap_estimate(bbox_a, bbox_b) -> float:
    lo = np.maximum(bbox_a[0], bbox_b[0])
    hi = np.minimum(bbox_a[1], bbox_b[1])
    inter = np.clip(hi - lo, 0, None)
    inter_vol = float(np.prod(inter))
    vol_a = float(np.prod(bbox_a[1] - bbox_a[0]))
    vol_b = float(np.prod(bbox_b[1] - bbox_b[0]))
    union = vol_a + vol_b - inter_vol
    return inter_vol / union if union > 0 else 0.0


def sagittal_pair_metrics(series_a: str, series_b: str) -> dict:
    ga, gb = series_geometry[series_a], series_geometry[series_b]
    sa, sb = ga["sample_ds"], gb["sample_ds"]
    row_a, col_a = row_column_cosines(sa.ImageOrientationPatient)
    row_b, col_b = row_column_cosines(sb.ImageOrientationPatient)
    normal_a = plane_normal(row_a, col_a)
    normal_b = plane_normal(row_b, col_b)

    cos_angle = float(np.clip(np.dot(normal_a, normal_b) / (np.linalg.norm(normal_a) * np.linalg.norm(normal_b)), -1.0, 1.0))
    orientation_angle_deg = float(np.degrees(np.arccos(abs(cos_angle))))  # 0 == parallel planes, direction-agnostic

    def centroid(sid):
        points = np.array([np.asarray(inst.dataset.ImagePositionPatient, dtype=np.float64) for _, inst in slice_ordering[sid]["ordered"]])
        return points.mean(axis=0)

    center_a, center_b = centroid(series_a), centroid(series_b)
    center_distance_mm = float(np.linalg.norm(center_a - center_b))

    corners_a, corners_b = series_corner_points(series_a), series_corner_points(series_b)
    bbox_a = (corners_a.min(axis=0), corners_a.max(axis=0))
    bbox_b = (corners_b.min(axis=0), corners_b.max(axis=0))
    overlap = bbox_overlap_estimate(bbox_a, bbox_b)

    same_for = bool(ga["for_uid_opaque"] is not None and ga["for_uid_opaque"] == gb["for_uid_opaque"])
    frame_status = classify_frame_relationship(ga["for_uid_opaque"], gb["for_uid_opaque"])
    spatial_relationship_dicom_validated = frame_status in (
        "DIRECT_DICOM_GEOMETRY_ALLOWED", "REGISTERED_DICOM_GEOMETRY_ALLOWED",
    )
    warn = []
    if not same_for:
        warn.append("different_frame_of_reference_uid_per_series")
    if orientation_angle_deg > 5.0:
        warn.append("sagittal_normals_not_closely_parallel")
    if not spatial_relationship_dicom_validated:
        warn.append("spatial_relationship_not_dicom_validated_numeric_only")

    return {
        "series_a": series_a,
        "series_b": series_b,
        "orientation_angle_deg": orientation_angle_deg,
        "center_distance_mm": center_distance_mm,
        "physical_overlap_estimate": overlap,
        "same_frame_of_reference": same_for,
        "frame_relationship_status": frame_status,
        # Numeric agreement of the two orientation matrices/positions, computed purely from each
        # series' own patient-space coordinates -- NOT proof that those coordinate systems are
        # the same physical space when FrameOfReferenceUID differs.
        "geometry_numerically_compatible": bool(orientation_angle_deg <= 5.0),
        "spatial_relationship_dicom_validated": spatial_relationship_dicom_validated,
        "warnings": "; ".join(warn),
    }


sagittal_pairing_rows = []
if REAL_DICOM_USED:
    sag_t1_ids = series_inventory.loc[series_inventory["candidate_role"] == "sagittal_t1", "series_opaque_id"].tolist()
    sag_t2_ids = series_inventory.loc[series_inventory["candidate_role"] == "sagittal_t2", "series_opaque_id"].tolist()
    for a in sag_t1_ids:
        for b in sag_t2_ids:
            sagittal_pairing_rows.append(sagittal_pair_metrics(a, b))

sagittal_pairing_metrics = pd.DataFrame(sagittal_pairing_rows)
sagittal_pairing_metrics


,series_a,series_b,orientation_angle_deg,center_distance_mm,physical_overlap_estimate,same_frame_of_reference,frame_relationship_status,geometry_numerically_compatible,spatial_relationship_dicom_validated,warnings
0,28ea4937243f,25aa08338722,0.0,0.0,1.0,False,DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED,True,False,different_frame_of_reference_uid_per_series; s...


## 8. Referencias anatómicas (sin inventar niveles)

Se admiten tres fuentes (`reference_source`): `existing_segmentation_reference`,
`manual_reference`, `geometric_test_reference`. En esta ejecución se usa **exclusivamente**
`manual_reference`: el punto central de un slice sagital intermedio (elegido por posición física,
no por índice arbitrario de lista), usado solo para validar la geometría de proyección. No se
afirma ningún nivel lumbar (`level_if_known = null`), consistente con que Notebook 66 no hace
localización automática de niveles — eso es tarea del Notebook 67.


In [14]:
references = []

if REAL_DICOM_USED and sag_t2_ids:
    reference_series = sag_t2_ids[0]
    ordered = slice_ordering[reference_series]["ordered"]
    mid_index = len(ordered) // 2
    mid_scalar, mid_inst = ordered[mid_index]
    ds = mid_inst.dataset
    mid_row, mid_col = int(ds.Rows) // 2, int(ds.Columns) // 2
    patient_xyz = pixel_to_patient_xyz(ds.ImagePositionPatient, ds.ImageOrientationPatient, ds.PixelSpacing, mid_row, mid_col)

    references.append({
        "reference_id": "manual_ref_sagittal_center_mid_slice",
        "reference_source": "manual_reference",
        "level_if_known": None,
        "source_series_role": "sagittal_t2",
        "source_series_opaque_id": reference_series,
        "source_slice_index_in_physical_order": mid_index,
        "row": mid_row,
        "col": mid_col,
        "patient_xyz": patient_xyz.tolist(),
        "notes": "Center pixel of the physically-middle sagittal T2 slice; heuristic anatomical "
                 "midline pick for geometry validation only, not a clinical or level-specific annotation.",
    })

references_df = pd.DataFrame(references)
references_df


,reference_id,reference_source,level_if_known,source_series_role,source_series_opaque_id,source_slice_index_in_physical_order,row,col,patient_xyz,notes
0,manual_ref_sagittal_center_mid_slice,manual_reference,None,sagittal_t2,25aa08338722,6,256,256,"[-8.196406432, 51.20052495999999, 34.054605760...",Center pixel of the physically-middle sagittal...


## 9. Sagittal → Axial: núcleo del notebook

Para cada punto de referencia se calcula la distancia al plano de cada slice axial candidato, se
seleccionan los N más cercanos, y se verifica si la proyección cae dentro del FOV real de la
imagen axial (no alcanza con estar cerca del plano).


In [15]:
def project_reference_to_axial_series(point_xyz, axial_series_id: str, top_n: int = 3) -> list[dict]:
    g = series_geometry[axial_series_id]
    candidates = []
    for s, inst in slice_ordering[axial_series_id]["ordered"]:
        ds = inst.dataset
        plane = build_plane(ds.ImagePositionPatient, ds.ImageOrientationPatient, ds.PixelSpacing, ds.Rows, ds.Columns)
        projection = project_patient_point_to_image_plane(point_xyz, plane)
        candidates.append({"physical_scalar": s, **projection})
    candidates.sort(key=lambda c: abs(c["distance_to_plane_mm"]))
    return candidates[:top_n]


axial_pairing_rows = []
axial_candidates_by_reference = {}

if REAL_DICOM_USED:
    axial_ids = series_inventory.loc[series_inventory["candidate_role"] == "axial_t2", "series_opaque_id"].tolist()
    for ref in references:
        for axial_id in axial_ids:
            candidates = project_reference_to_axial_series(ref["patient_xyz"], axial_id, top_n=5)
            axial_candidates_by_reference[(ref["reference_id"], axial_id)] = candidates
            if not candidates:
                continue
            best = candidates[0]
            second = candidates[1] if len(candidates) > 1 else None
            under_2mm = sum(1 for c in candidates if abs(c["distance_to_plane_mm"]) < 2.0)
            under_5mm = sum(1 for c in candidates if abs(c["distance_to_plane_mm"]) < 5.0)

            axial_pairing_rows.append({
                "reference_id": ref["reference_id"],
                "axial_series_opaque_id": axial_id,
                "best_distance_mm": abs(best["distance_to_plane_mm"]),
                "best_row_float": best["row_float"],
                "best_col_float": best["col_float"],
                "best_inside_fov": best["inside_fov"],
                "second_best_distance_mm": abs(second["distance_to_plane_mm"]) if second else None,
                "candidate_count_under_2mm": under_2mm,
                "candidate_count_under_5mm": under_5mm,
            })

axial_pairing_metrics = pd.DataFrame(axial_pairing_rows)
axial_pairing_metrics


,reference_id,axial_series_opaque_id,best_distance_mm,best_row_float,best_col_float,best_inside_fov,second_best_distance_mm,candidate_count_under_2mm,candidate_count_under_5mm
0,manual_ref_sagittal_center_mid_slice,b65eca7cfbbc,0.646663,290.695075,232.250685,True,4.853312,1,2


## 10. Geometry confidence (score determinístico, NO una probabilidad)

> This is a deterministic geometry quality score, not a calibrated probability.

Fórmula exacta (pesos suman 1.0):

```
geometry_confidence =
    0.15 * metadata_score
  + 0.15 * orientation_score
  + 0.10 * frame_relationship_score
  + 0.30 * distance_score
  + 0.20 * fov_score
  + 0.10 * spacing_score
```

- `metadata_score`: 1.0 si ambas series (sagital de origen y axial candidata) tienen todos los
  campos DICOM de geometría requeridos; 0.0 si falta alguno.
- `orientation_score`: 1.0 si `row_cosines`/`column_cosines` de ambas series son ortonormales
  dentro de tolerancia (mismo chequeo que GATE B); 0.0 si no.
- `frame_relationship_score`: 1.0 si `frame_relationship_status == DIRECT_DICOM_GEOMETRY_ALLOWED`,
  0.5 si `REGISTERED_DICOM_GEOMETRY_ALLOWED`, 0.0 si `DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED`.
- `distance_score`: **solo se otorga si `distance_validated=True`** (frame relationship A o B).
  Si `distance_validated=False`, `distance_score=0.0` sin importar qué tan chica sea la
  distancia cruda — una distancia numérica entre dos frames no relacionados no es evidencia de
  cercanía física real. Cuando validada: 1.0 si `< 2mm`, 0.7 si `< 5mm`, 0.3 si `< 10mm`, 0.0 si no.
- `fov_score`: 1.0 si la proyección cae dentro del FOV, 0.0 si no.
- `spacing_score`: 1.0 si el spacing de slices es regular (`spacing_std/median <= 0.15`), 0.5 si
  levemente irregular, 0.0 si `irregular_spacing` es `True` con relación `> 0.5`.


In [16]:
def geometry_confidence_score(metadata_ok: bool, orientation_ok: bool, frame_relationship_status: str,
                               distance_mm: float, distance_validated: bool, inside_fov: bool,
                               irregular_spacing) -> dict:
    metadata_score = 1.0 if metadata_ok else 0.0
    orientation_score = 1.0 if orientation_ok else 0.0
    if frame_relationship_status == "DIRECT_DICOM_GEOMETRY_ALLOWED":
        frame_relationship_score = 1.0
    elif frame_relationship_status == "REGISTERED_DICOM_GEOMETRY_ALLOWED":
        frame_relationship_score = 0.5
    else:
        frame_relationship_score = 0.0

    if not distance_validated:
        # Cross-frame, unregistered: never credit proximity as evidence of pairing quality.
        distance_score = 0.0
    elif distance_mm < 2.0:
        distance_score = 1.0
    elif distance_mm < 5.0:
        distance_score = 0.7
    elif distance_mm < 10.0:
        distance_score = 0.3
    else:
        distance_score = 0.0

    fov_score = 1.0 if inside_fov else 0.0
    if irregular_spacing is False:
        spacing_score = 1.0
    elif irregular_spacing is None:
        spacing_score = 0.5
    else:
        spacing_score = 0.0

    confidence = (
        0.15 * metadata_score + 0.15 * orientation_score + 0.10 * frame_relationship_score
        + 0.30 * distance_score + 0.20 * fov_score + 0.10 * spacing_score
    )
    return {
        "metadata_score": metadata_score, "orientation_score": orientation_score,
        "frame_relationship_score": frame_relationship_score, "distance_score": distance_score,
        "fov_score": fov_score, "spacing_score": spacing_score,
        "geometry_confidence": round(confidence, 4),
    }


print("geometry_confidence_score defined. NOTE: this is a deterministic geometry quality score,")
print("not a calibrated probability. Distance is only credited when distance_validated=True.")


geometry_confidence_score defined. NOTE: this is a deterministic geometry quality score,
not a calibrated probability. Distance is only credited when distance_validated=True.


## 11. GATE B / GATE D en código (orientation validity + slice ordering)


In [17]:
orientation_checks = []
if REAL_DICOM_USED:
    for _, row in series_inventory.iterrows():
        sid = row["series_opaque_id"]
        sample = series_geometry[sid]["sample_ds"]
        if not hasattr(sample, "ImageOrientationPatient"):
            orientation_checks.append({"series_opaque_id": sid, "orientation_valid": False})
            continue
        row_cos, col_cos = row_column_cosines(sample.ImageOrientationPatient)
        normal = plane_normal(row_cos, col_cos)
        norm_ok = abs(np.linalg.norm(row_cos) - 1) < 1e-2 and abs(np.linalg.norm(col_cos) - 1) < 1e-2
        orth_ok = abs(np.dot(row_cos, col_cos)) < 1e-2
        normal_ok = abs(np.linalg.norm(normal) - 1) < 1e-2
        orientation_checks.append({
            "series_opaque_id": sid,
            "orientation_valid": bool(norm_ok and orth_ok and normal_ok),
        })

orientation_checks_df = pd.DataFrame(orientation_checks)
ORIENTATION_VALID_ALL = bool(orientation_checks_df["orientation_valid"].all()) if len(orientation_checks_df) else False
orientation_checks_df


,series_opaque_id,orientation_valid
0,28ea4937243f,True
1,b65eca7cfbbc,True
2,25aa08338722,True


In [18]:
SLICE_ORDERING_PASS = True
if REAL_DICOM_USED:
    for sid, info in slice_ordering.items():
        if info["duplicates"] not in (0, None) or not info["monotonic"]:
            SLICE_ORDERING_PASS = False
            warnings.append(f"Slice ordering issue in series {sid}: duplicates={info['duplicates']}, monotonic={info['monotonic']}")
else:
    SLICE_ORDERING_PASS = False

print("SLICE_ORDERING_PASS:", SLICE_ORDERING_PASS)


SLICE_ORDERING_PASS: True


## 12. `LevelSeriesBundle` experimental

Se genera solo cuando existe una referencia con `level_if_known` **conocido**. En esta ejecución
la única referencia disponible es `manual_reference` con `level_if_known = null`, por lo que
**no** se emite ningún `LevelSeriesBundle` real — `post_e50_level_series_bundles.json` queda como
`[]`. El esquema experimental se valida por separado con un caso sintético que **nunca** se mezcla
con artefactos derivados del estudio real: se escribe en
`artifacts/post_e50/series_pairing/schema_test_fixtures/level_series_bundle_example.json`,
marcado explícitamente `"synthetic": true, "testOnly": true`.


In [19]:
LEVEL_SERIES_BUNDLE_SCHEMA_VERSION = "pfi.post-e50.level-series-bundle.v0"
SCHEMA_FIXTURES_DIR = PAIRING_DIR / "schema_test_fixtures"


def build_level_series_bundle(study_opaque_id, level, reference, sag_t1_info, sag_t2_info, axial_info, pairing_confidence) -> dict:
    return {
        "schemaVersion": LEVEL_SERIES_BUNDLE_SCHEMA_VERSION,
        "studyOpaqueId": study_opaque_id,
        "level": level,
        "reference": {"source": reference["reference_source"], "patientXYZ": reference["patient_xyz"]},
        "series": {
            "sagittalT1": sag_t1_info,
            "sagittalT2": sag_t2_info,
            "axialT2": axial_info,
        },
        "pairing": {
            "method": "dicom_patient_geometry",
            "sameFrameOfReference": bool(sag_t1_info.get("sameFrameOfReference")) if sag_t1_info else None,
            "geometryValid": bool(ORIENTATION_VALID_ALL and SLICE_ORDERING_PASS),
            "confidence": pairing_confidence,
        },
    }


# Real bundles: only ever populated from references with a confidently known level. Currently
# always empty for this run because the only reference used is manual_reference/level=null.
level_series_bundles: list[dict] = []
known_level_references = [r for r in references if r["level_if_known"] is not None]
if not known_level_references:
    limitations.append(
        "No LevelSeriesBundle was emitted with a known lumbar level: the only reference used "
        "(manual_reference) intentionally has level_if_known=null. Automatic level naming is "
        "explicitly out of scope for Notebook 66 (deferred to Notebook 67)."
    )
for ref in known_level_references:
    raise RuntimeError("Unexpected known-level reference: level-bundle construction from real data is not implemented in this run.")

# Synthetic example to prove the schema round-trips (geometric_test_reference; never claims a
# real level). Kept fully separate from real-study artifacts.
synthetic_bundle = build_level_series_bundle(
    study_opaque_id="synthetic_test_study",
    level="L4-L5",
    reference={"reference_source": "geometric_test_reference", "patient_xyz": [0.0, 0.0, 0.0]},
    sag_t1_info={"available": True, "seriesOpaqueId": "synthetic_sag_t1", "referenceSlice": 5},
    sag_t2_info={"available": True, "seriesOpaqueId": "synthetic_sag_t2", "referenceSlice": 6, "sameFrameOfReference": True},
    axial_info={"available": True, "seriesOpaqueId": "synthetic_axial_t2", "matchedSlices": [14, 15, 16], "distancesMm": [0.5, 1.2, 2.1]},
    pairing_confidence=0.91,
)
synthetic_bundle["synthetic"] = True
synthetic_bundle["testOnly"] = True
safe_write_text(
    SCHEMA_FIXTURES_DIR / "level_series_bundle_example.json",
    json.dumps(synthetic_bundle, indent=2, ensure_ascii=False),
)
print("LevelSeriesBundle schema validated with one synthetic example, written to schema_test_fixtures/.")
print("Real-study bundles with a known level:", len(known_level_references), "(post_e50_level_series_bundles.json will be [])")


LevelSeriesBundle schema validated with one synthetic example, written to schema_test_fixtures/.
Real-study bundles with a known level: 0 (post_e50_level_series_bundles.json will be [])


## 13. Visualizaciones


In [20]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 1. Series inventory & orientation
fig, ax = plt.subplots(figsize=(8, 4))
if len(series_inventory):
    ax.bar(series_inventory["candidate_role"], series_inventory["slice_count"])
    ax.set_ylabel("slice_count")
    ax.set_title("Series inventory: slice count by candidate_role")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "series_inventory_roles.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_36224\176014284.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# 2. Simple 3D-ish plot: sagittal plane origins, axial plane origins, reference point
fig, ax = plt.subplots(figsize=(6, 6))
if REAL_DICOM_USED:
    for sid in sag_t1_ids + sag_t2_ids:
        pts = np.array([np.asarray(inst.dataset.ImagePositionPatient) for _, inst in slice_ordering[sid]["ordered"]])
        ax.scatter(pts[:, 0], pts[:, 2], s=8, label=f"sagittal[{sid[:6]}]")
    for sid in axial_ids:
        pts = np.array([np.asarray(inst.dataset.ImagePositionPatient) for _, inst in slice_ordering[sid]["ordered"]])
        ax.scatter(pts[:, 0], pts[:, 2], s=8, marker="x", label=f"axial[{sid[:6]}]")
    for ref in references:
        xyz = ref["patient_xyz"]
        ax.scatter([xyz[0]], [xyz[2]], s=120, marker="*", color="red", label=ref["reference_id"][:20])
ax.set_xlabel("Patient X (mm)")
ax.set_ylabel("Patient Z (mm)")
ax.set_title("Series geometry (X-Z projection): sagittal/axial slice origins + reference point")
ax.legend(fontsize=6, loc="best")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "series_geometry_3d.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_36224\3453020339.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# 3. Sagittal view showing the reference point pixel location
if REAL_DICOM_USED and references:
    ref = references[0]
    ref_series = ref["source_series_opaque_id"]
    _, ref_inst = slice_ordering[ref_series]["ordered"][ref["source_slice_index_in_physical_order"]]
    pixel_array = load_one_pixel_array(DICOM_SOURCE_PATH, ref_inst.entry_name)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(pixel_array, cmap="gray")
    ax.scatter([ref["col"]], [ref["row"]], c="red", marker="+", s=200)
    ax.set_title("Sagittal reference slice (manual_reference point)")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "sagittal_reference_point.png", dpi=120)
    plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_36224\335237434.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# 4. Closest axial slice showing where the same XYZ projects
if REAL_DICOM_USED and references and axial_candidates_by_reference:
    ref = references[0]
    (ref_id, axial_id), candidates = next(iter(axial_candidates_by_reference.items()))
    if candidates:
        best = candidates[0]
        _, best_inst = slice_ordering[axial_id]["ordered"][
            [s for s, _ in slice_ordering[axial_id]["ordered"]].index(best["physical_scalar"])
        ]
        pixel_array = load_one_pixel_array(DICOM_SOURCE_PATH, best_inst.entry_name)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(pixel_array, cmap="gray")
        if best["inside_fov"]:
            ax.scatter([best["col_float"]], [best["row_float"]], c="red", marker="+", s=200)
        ax.set_title(f"Closest axial slice (dist={abs(best['distance_to_plane_mm']):.2f} mm, inside_fov={best['inside_fov']})")
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "axial_closest_slice_projection.png", dpi=120)
        plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_36224\3440037958.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# 5. axial slice index vs distance_to_reference_mm
if REAL_DICOM_USED and axial_candidates_by_reference:
    (ref_id, axial_id), candidates = next(iter(axial_candidates_by_reference.items()))
    ordered = slice_ordering[axial_id]["ordered"]
    all_distances = []
    for i, (s, inst) in enumerate(ordered):
        ds = inst.dataset
        plane = build_plane(ds.ImagePositionPatient, ds.ImageOrientationPatient, ds.PixelSpacing, ds.Rows, ds.Columns)
        d = signed_distance_point_to_plane(references[0]["patient_xyz"], plane)
        all_distances.append((i, abs(d)))
    xs, ys = zip(*all_distances)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(xs, ys, marker="o")
    ax.set_xlabel("axial slice index (physical order)")
    ax.set_ylabel("distance_to_reference_mm")
    ax.set_title("Axial distance profile to reference point")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "axial_distance_profile.png", dpi=120)
    plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_36224\637663101.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
# 6. Table: top N matched axial slices per reference
fig, ax = plt.subplots(figsize=(7, 3))
ax.axis("off")
if REAL_DICOM_USED and axial_candidates_by_reference:
    (ref_id, axial_id), candidates = next(iter(axial_candidates_by_reference.items()))
    table_rows = [[i, f"{abs(c['distance_to_plane_mm']):.3f}", c["inside_fov"]] for i, c in enumerate(candidates)]
    tbl = ax.table(cellText=table_rows, colLabels=["rank", "distance_mm", "inside_fov"], loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.6)
    ax.set_title(f"Top {len(candidates)} matched axial slices for {ref_id}", pad=16)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pairing_examples_top_matches.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_36224\2731004404.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. `pairing_metrics` consolidada


In [26]:
pairing_metrics_rows = []
if REAL_DICOM_USED:
    for ref in references:
        for axial_id in axial_ids:
            candidates = axial_candidates_by_reference.get((ref["reference_id"], axial_id), [])
            if not candidates:
                continue
            best = candidates[0]
            second = candidates[1] if len(candidates) > 1 else None
            sag_id = ref["source_series_opaque_id"]
            same_for = series_geometry[sag_id]["for_uid_opaque"] == series_geometry[axial_id]["for_uid_opaque"]
            frame_status = classify_frame_relationship(series_geometry[sag_id]["for_uid_opaque"], series_geometry[axial_id]["for_uid_opaque"])
            distance_validated = frame_status in ("DIRECT_DICOM_GEOMETRY_ALLOWED", "REGISTERED_DICOM_GEOMETRY_ALLOWED")
            distance_interpretation = "REGISTERED_OR_SAME_FRAME" if distance_validated else "UNREGISTERED_CROSS_FRAME_EXPLORATORY"
            metadata_ok = bool(series_inventory.loc[series_inventory["series_opaque_id"] == sag_id, "geometry_valid"].iloc[0]) and \
                          bool(series_inventory.loc[series_inventory["series_opaque_id"] == axial_id, "geometry_valid"].iloc[0])
            orientation_ok = bool(
                orientation_checks_df.loc[orientation_checks_df["series_opaque_id"] == sag_id, "orientation_valid"].iloc[0]
                and orientation_checks_df.loc[orientation_checks_df["series_opaque_id"] == axial_id, "orientation_valid"].iloc[0]
            )
            irregular = slice_ordering[axial_id]["irregular_spacing"]
            raw_distance_mm = abs(best["distance_to_plane_mm"])
            scores = geometry_confidence_score(
                metadata_ok, orientation_ok, frame_status, raw_distance_mm, distance_validated, best["inside_fov"], irregular,
            )
            under_2mm = sum(1 for c in candidates if abs(c["distance_to_plane_mm"]) < 2.0)
            under_5mm = sum(1 for c in candidates if abs(c["distance_to_plane_mm"]) < 5.0)
            row_warn = []
            if not same_for:
                row_warn.append("different_frame_of_reference")
            if not distance_validated:
                row_warn.append("distance_not_dicom_validated_exploratory_only")
            if not best["inside_fov"]:
                row_warn.append("best_candidate_outside_fov")

            pairing_metrics_rows.append({
                "study_opaque_id": STUDY_OPAQUE_ID,
                "reference_id": ref["reference_id"],
                "level_if_known": ref["level_if_known"],
                "reference_source": ref["reference_source"],
                "sagittal_source_role": ref["source_series_role"],
                "axial_series_opaque_id": axial_id,
                "best_axial_slice_physical_index": [s for s, _ in slice_ordering[axial_id]["ordered"]].index(best["physical_scalar"]),
                # raw_coordinate_distance_mm: exploratory only, see distance_validated below.
                "raw_coordinate_distance_mm": raw_distance_mm,
                "best_distance_mm": raw_distance_mm,
                "distance_validated": distance_validated,
                "distance_interpretation": distance_interpretation,
                "second_best_distance_mm": abs(second["distance_to_plane_mm"]) if second else None,
                "candidate_count_under_2mm": under_2mm,
                "candidate_count_under_5mm": under_5mm,
                "inside_fov": best["inside_fov"],
                "same_frame_of_reference": same_for,
                "frame_relationship_status": frame_status,
                "orientation_valid": orientation_ok,
                "spacing_regular": (irregular is False),
                "geometry_confidence": scores["geometry_confidence"],
                "status": "matched" if (best["inside_fov"] and distance_validated) else (
                    "matched_outside_fov" if not best["inside_fov"] else "matched_unregistered_cross_frame"
                ),
                "warnings": "; ".join(row_warn),
            })

pairing_metrics = pd.DataFrame(pairing_metrics_rows)
pairing_metrics


,study_opaque_id,reference_id,level_if_known,reference_source,sagittal_source_role,axial_series_opaque_id,best_axial_slice_physical_index,raw_coordinate_distance_mm,best_distance_mm,distance_validated,...,candidate_count_under_2mm,candidate_count_under_5mm,inside_fov,same_frame_of_reference,frame_relationship_status,orientation_valid,spacing_regular,geometry_confidence,status,warnings
0,206aa67ee4e6,manual_ref_sagittal_center_mid_slice,None,manual_reference,sagittal_t2,b65eca7cfbbc,12,0.646663,0.646663,False,...,1,2,True,False,DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED,True,False,0.5,matched_unregistered_cross_frame,different_frame_of_reference; distance_not_dic...


## 15. Quality gates (A–F)

**GATE E fue rediseñado** tras el hallazgo de `FrameOfReferenceUID` distinto entre las 3 series
(Sección 2b) — ya no basta con que la distancia numérica sea finita:

- **PASS**: mismo `FrameOfReference`, o transformación DICOM explícita validada entre frames.
- **PARTIAL**: las transformaciones matemáticas funcionan (distancias finitas, proyección
  calculable) pero los frames son distintos y no existe transformación registrada entre ellos.
- **FAIL**: metadata insuficiente o transformaciones inválidas.

El resto de los gates (A–D, F) permanecen binarios PASS/FAIL.


In [27]:
gate_status: dict[str, str] = {}

gate_status["GATE_A_dicom_metadata"] = "PASS" if bool(
    REAL_DICOM_USED and len(series_inventory) and series_inventory["geometry_valid"].any()
) else "FAIL"
gate_status["GATE_B_orientation"] = "PASS" if ORIENTATION_VALID_ALL else "FAIL"
gate_status["GATE_C_transform_roundtrip"] = "PASS" if bool(GEOMETRY_UNIT_TESTS_PASS and max_roundtrip_error <= 1e-3) else "FAIL"
gate_status["GATE_D_slice_ordering"] = "PASS" if SLICE_ORDERING_PASS else "FAIL"


def evaluate_gate_e(pairing_df: pd.DataFrame) -> str:
    if len(pairing_df) == 0:
        return "FAIL"
    if not np.isfinite(pairing_df["raw_coordinate_distance_mm"]).all():
        return "FAIL"
    if bool(pairing_df["distance_validated"].any()):
        return "PASS"
    return "PARTIAL"


gate_status["GATE_E_cross_series_spatial_relationship"] = evaluate_gate_e(pairing_metrics)
gate_status["GATE_F_privacy"] = None  # evaluated in the privacy audit cell below

gates_df = pd.DataFrame(sorted(gate_status.items()), columns=["gate", "status"])
gates_df


,gate,status
0,GATE_A_dicom_metadata,PASS
1,GATE_B_orientation,PASS
2,GATE_C_transform_roundtrip,PASS
3,GATE_D_slice_ordering,PASS
4,GATE_E_cross_series_spatial_relationship,PARTIAL
5,GATE_F_privacy,None


## 16. Privacy audit

Se verifica, sobre las estructuras que están a punto de escribirse a disco, que ningún valor
crudo de `PatientName`, `PatientID`, `AccessionNumber`, `StudyInstanceUID`,
`SeriesInstanceUID`, `SOPInstanceUID` o rutas locales (`C:\Users\`) esté presente. Los UID reales
solo existieron en memoria durante la ejecución para calcular los IDs opacos.


In [28]:
forbidden_values = set()
if REAL_DICOM_USED:
    for inst in instances:
        ds = inst.dataset
        for field_name in FORBIDDEN_IDENTIFIER_FIELDS:
            value = getattr(ds, field_name, None)
            if value:
                forbidden_values.add(str(value))
safe_write_text._forbidden_values = forbidden_values

candidate_outputs = {
    "series_inventory": series_inventory.to_csv(index=False),
    "sagittal_pairing_metrics": sagittal_pairing_metrics.to_csv(index=False) if len(sagittal_pairing_metrics) else "",
    "axial_pairing_metrics": axial_pairing_metrics.to_csv(index=False) if len(axial_pairing_metrics) else "",
    "pairing_metrics": pairing_metrics.to_csv(index=False) if len(pairing_metrics) else "",
    "level_series_bundles": json.dumps(level_series_bundles, indent=2),
}

privacy_findings = []
for name, text in candidate_outputs.items():
    for value in forbidden_values:
        if value in text:
            privacy_findings.append(f"{name} contains a raw forbidden identifier value")
    if "C:\\Users\\" in text or "/Users/" in text:
        privacy_findings.append(f"{name} contains a local filesystem path")

GATE_F_PRIVACY_PASS = len(privacy_findings) == 0
gate_status["GATE_F_privacy"] = "PASS" if GATE_F_PRIVACY_PASS else "FAIL"
gates_df.loc[gates_df["gate"] == "GATE_F_privacy", "status"] = gate_status["GATE_F_privacy"]

print("Privacy findings:", privacy_findings if privacy_findings else "(none)")
print("GATE_F_privacy:", gate_status["GATE_F_privacy"])
if not GATE_F_PRIVACY_PASS:
    warnings.append(f"Privacy audit found issues: {privacy_findings}")


def overall_gate_status(statuses: list[str]) -> str:
    if any(s == "FAIL" for s in statuses):
        return "FAIL"
    if any(s == "PARTIAL" for s in statuses):
        return "PARTIAL"
    return "PASS"


QUALITY_GATE_OVERALL = overall_gate_status(list(gate_status.values()))
print()
print(gates_df)
print()
print("Overall quality gate:", QUALITY_GATE_OVERALL)


Privacy findings: (none)
GATE_F_privacy: PASS

                                       gate   status
0                     GATE_A_dicom_metadata     PASS
1                        GATE_B_orientation     PASS
2                GATE_C_transform_roundtrip     PASS
3                     GATE_D_slice_ordering     PASS
4  GATE_E_cross_series_spatial_relationship  PARTIAL
5                            GATE_F_privacy     PASS

Overall quality gate: PARTIAL


## 17. Artefactos de salida

Todo se escribe exclusivamente dentro de `artifacts/post_e50/series_pairing/` (incluida la
subcarpeta `figures/`, ya poblada en la sección 13) y `reports/post_e50/`.


In [29]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


series_inventory.to_csv(PAIRING_DIR / "post_e50_series_inventory.csv", index=False)

series_geometry_export = {
    "generated_at": GENERATED_AT,
    "study_opaque_id": STUDY_OPAQUE_ID if REAL_DICOM_USED else None,
    "real_dicom_used": REAL_DICOM_USED,
    "series": json.loads(series_inventory.to_json(orient="records")) if len(series_inventory) else [],
    "slice_ordering_summary": json.loads(slice_ordering_summary.to_json(orient="records")) if len(slice_ordering_summary) else [],
    "geometry_unit_tests": geometry_test_results,
    "max_roundtrip_error": max_roundtrip_error,
}
safe_write_text(
    PAIRING_DIR / "post_e50_series_geometry.json",
    json.dumps(series_geometry_export, indent=2, default=_json_default, ensure_ascii=False),
)

sagittal_pairing_metrics.to_csv(PAIRING_DIR / "post_e50_sagittal_pairing_metrics.csv", index=False)
if len(axial_pairing_metrics):
    axial_pairing_metrics.to_csv(PAIRING_DIR / "post_e50_axial_pairing_metrics.csv", index=False)
else:
    safe_write_text(PAIRING_DIR / "post_e50_axial_pairing_metrics.csv", "no_axial_candidates_found\n")

safe_write_text(
    PAIRING_DIR / "post_e50_level_series_bundles.json",
    json.dumps(level_series_bundles, indent=2, default=_json_default, ensure_ascii=False),
)

safe_write_text(
    PAIRING_DIR / "post_e50_geometry_quality_gates.json",
    json.dumps({"gates": gate_status, "overall_status": QUALITY_GATE_OVERALL, "warnings": warnings}, indent=2, ensure_ascii=False),
)

pairing_summary = {
    "generated_at": GENERATED_AT,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "real_dicom_used": REAL_DICOM_USED,
    "study_opaque_id": STUDY_OPAQUE_ID if REAL_DICOM_USED else None,
    "series_discovered": len(series_inventory),
    "sagittal_t1_found": bool((series_inventory["candidate_role"] == "sagittal_t1").any()) if len(series_inventory) else False,
    "sagittal_t2_found": bool((series_inventory["candidate_role"] == "sagittal_t2").any()) if len(series_inventory) else False,
    "axial_t2_found": bool((series_inventory["candidate_role"] == "axial_t2").any()) if len(series_inventory) else False,
    "axial_orientation_cluster_count": int(len(axial_orientation_clusters)),
    "pairing_metrics": json.loads(pairing_metrics.to_json(orient="records")) if len(pairing_metrics) else [],
    "gates": gate_status,
    "quality_gate_overall": QUALITY_GATE_OVERALL,
    "warnings": warnings,
    "limitations": limitations,
}
safe_write_text(
    PAIRING_DIR / "post_e50_pairing_summary.json",
    json.dumps(pairing_summary, indent=2, default=_json_default, ensure_ascii=False),
)

pairing_metrics.to_csv(PAIRING_DIR / "post_e50_pairing_metrics_full.csv", index=False)

print("Written:")
for f in sorted(PAIRING_DIR.rglob("post_e50_*")):
    print(" -", f.relative_to(REPO_ROOT))
for f in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", f.relative_to(REPO_ROOT))


Written:
 - artifacts\post_e50\series_pairing\post_e50_axial_orientation_clusters.csv
 - artifacts\post_e50\series_pairing\post_e50_axial_pairing_metrics.csv
 - artifacts\post_e50\series_pairing\post_e50_geometry_quality_gates.json
 - artifacts\post_e50\series_pairing\post_e50_level_series_bundles.json
 - artifacts\post_e50\series_pairing\post_e50_pairing_metrics_full.csv
 - artifacts\post_e50\series_pairing\post_e50_pairing_summary.json
 - artifacts\post_e50\series_pairing\post_e50_sagittal_pairing_metrics.csv
 - artifacts\post_e50\series_pairing\post_e50_series_geometry.json
 - artifacts\post_e50\series_pairing\post_e50_series_inventory.csv
 - artifacts\post_e50\series_pairing\figures\axial_closest_slice_projection.png
 - artifacts\post_e50\series_pairing\figures\axial_distance_profile.png
 - artifacts\post_e50\series_pairing\figures\axial_orientation_clusters.png
 - artifacts\post_e50\series_pairing\figures\pairing_examples_top_matches.png
 - artifacts\post_e50\series_pairing\figure

## 18. Reporte (Markdown)


In [30]:
report_lines = []
report_lines.append("# Post-E50 Multiseries Geometry Pairing")
report_lines.append("")
report_lines.append("## Objective")
report_lines.append("")
report_lines.append(
    "Build and validate a common spatial representation for Sagittal T1, Sagittal T2 and Axial "
    "T2 series of the same lumbar study, using DICOM patient-coordinate geometry, without "
    "inventing lumbar level labels."
)
report_lines.append("")
report_lines.append("## Scope")
report_lines.append("")
report_lines.append(
    "This notebook does geometry only: series classification, coordinate transforms, physical "
    "slice ordering, sagittal-sagittal compatibility, and sagittal-to-axial point projection. It "
    "does not train models, does not perform automatic level naming, and does not change "
    "`AUTOMATIC_DISC_LOCALIZATION_VALIDATED` in product code."
)
report_lines.append("")
report_lines.append("## Data source")
report_lines.append("")
report_lines.append(f"- Real DICOM used: `{REAL_DICOM_USED}`")
report_lines.append(f"- Discovery method: `{DICOM_SOURCE_METHOD.split(':')[0]}`")
report_lines.append(f"- ZIP SHA-256: `{dicom_zip_sha256}`")
report_lines.append(f"- Matches historically expected hash: `{dicom_zip_sha256 == EXPECTED_STUDY_ZIP_SHA256 if dicom_zip_sha256 else None}`")
report_lines.append("- The DICOM file was read directly from the ZIP in memory; nothing was copied into the repository.")
report_lines.append("")
report_lines.append("## Privacy")
report_lines.append("")
report_lines.append(
    "All persisted outputs use opaque IDs (`sha256(uid)[:12]`) instead of raw "
    "StudyInstanceUID/SeriesInstanceUID/SOPInstanceUID/PatientID/PatientName/AccessionNumber. "
    f"Privacy audit result: `{'PASS' if GATE_F_PRIVACY_PASS else 'FAIL'}`."
)
report_lines.append("")
report_lines.append("## DICOM geometry methodology")
report_lines.append("")
report_lines.append(
    "`ImageOrientationPatient` is split into `row_cosines` (first 3 values, direction of "
    "increasing column index) and `column_cosines` (last 3 values, direction of increasing row "
    "index), following pydicom's own naming to avoid the row/column swap pitfall. "
    "`patient = ImagePositionPatient + col*PixelSpacing[1]*row_cosines + row*PixelSpacing[0]*column_cosines`. "
    "`normal = cross(row_cosines, column_cosines)`. "
    "**Pixel/patient transforms are validated** (Section: Geometry unit tests) and are correct "
    "**within** the FrameOfReference of a single series. **Within-series DICOM geometry is "
    "validated**: orientation matrices are orthonormal (GATE B) and slices order physically "
    "without duplicates (GATE D)."
)
report_lines.append("")
report_lines.append("## Cross-series Frame of Reference")
report_lines.append("")
report_lines.append(
    f"This study's three series (`sagittal_t1`, `sagittal_t2`, `axial_t2`) use **different "
    f"`FrameOfReferenceUID` values** ({len(registration_objects_found)} explicit DICOM "
    "Spatial/Deformable Registration object(s) found by read-only search). Direct cross-series "
    "coordinate comparison requires either a matching FrameOfReference or an explicit DICOM "
    "registration between frames; neither is present here. Therefore **direct cross-series "
    "correspondence between these series remains unvalidated** (`DIRECT_CROSS_SERIES_MAPPING_NOT_VALIDATED`) "
    "until registration is resolved -- the raw coordinate distances computed in this notebook "
    "are exploratory only (`UNREGISTERED_CROSS_FRAME_EXPLORATORY`) and were **not** used "
    "positively in `geometry_confidence`."
)
report_lines.append("")
report_lines.append("## Axial orientation clustering")
report_lines.append("")
report_lines.append(
    f"The Axial T2 series' per-slice `ImageOrientationPatient` was NOT constant across its "
    f"{int(series_inventory.loc[series_inventory['candidate_role']=='axial_t2','slice_count'].iloc[0]) if len(series_inventory) and (series_inventory['candidate_role']=='axial_t2').any() else 'N/A'} "
    f"slices. Clustering by angular distance between plane normals (tolerance "
    f"{ORIENTATION_CLUSTER_ANGLE_TOLERANCE_DEG} deg) found **{len(axial_orientation_clusters)} distinct "
    "orientation cluster(s)**, each ordered physically within itself rather than as one 24-slice "
    "stack. See `post_e50_axial_orientation_clusters.csv` and `figures/axial_orientation_clusters.png`. "
    "No cluster is claimed to correspond to a specific lumbar level -- that determination belongs "
    "to a future level-localization notebook."
)
report_lines.append(axial_orientation_clusters.to_markdown(index=False) if len(axial_orientation_clusters) else "_No clusters computed._")
report_lines.append("")
report_lines.append("## Series discovery")
report_lines.append("")
report_lines.append(series_inventory.to_markdown(index=False) if len(series_inventory) else "_No series discovered (no real DICOM source)._")
report_lines.append("")
report_lines.append("## Coordinate transformations")
report_lines.append("")
report_lines.append("Implemented: `pixel_to_patient_xyz`, `patient_xyz_to_slice_coordinates`, `build_plane`, "
                     "`signed_distance_point_to_plane`, `absolute_distance_point_to_plane`, "
                     "`project_patient_point_to_image_plane`. See notebook Section 3 for the exact formulas.")
report_lines.append("")
report_lines.append("## Geometry unit tests")
report_lines.append("")
for name, passed in geometry_test_results.items():
    report_lines.append(f"- `{name}` -> `{'PASS' if passed else 'FAIL'}`")
report_lines.append(f"- max_roundtrip_error = `{max_roundtrip_error}`")
report_lines.append("")
report_lines.append("## Sagittal T1 / Sagittal T2 compatibility")
report_lines.append("")
report_lines.append(sagittal_pairing_metrics.to_markdown(index=False) if len(sagittal_pairing_metrics) else "_No sagittal T1/T2 pair found._")
report_lines.append("")
report_lines.append("## Sagittal / Axial pairing")
report_lines.append("")
report_lines.append(pairing_metrics.to_markdown(index=False) if len(pairing_metrics) else "_No pairing metrics generated._")
report_lines.append("")
report_lines.append("## Pairing examples")
report_lines.append("")
report_lines.append("See `figures/axial_closest_slice_projection.png`, `figures/axial_distance_profile.png`, "
                     "and `figures/pairing_examples_top_matches.png`.")
report_lines.append("")
report_lines.append("## Geometry quality score")
report_lines.append("")
report_lines.append(
    "`geometry_confidence` is a deterministic, documented weighted combination (Section 10 of "
    "the notebook) — explicitly **not** a calibrated probability."
)
report_lines.append("")
report_lines.append("## Quality gates")
report_lines.append("")
report_lines.append(gates_df.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Results")
report_lines.append("")
report_lines.append(f"- Sagittal T1 found: `{pairing_summary['sagittal_t1_found']}`")
report_lines.append(f"- Sagittal T2 found: `{pairing_summary['sagittal_t2_found']}`")
report_lines.append(f"- Axial T2 found: `{pairing_summary['axial_t2_found']}`")
report_lines.append(f"- Axial orientation clusters found: `{len(axial_orientation_clusters)}`")
report_lines.append(f"- Quality gate overall: `{QUALITY_GATE_OVERALL}`")
report_lines.append("")
report_lines.append("## Failure cases")
report_lines.append("")
if not len(pairing_metrics):
    report_lines.append("- No sagittal-to-axial pairing could be computed for this run.")
else:
    outside_fov = pairing_metrics[~pairing_metrics["inside_fov"]]
    if len(outside_fov):
        report_lines.append(f"- {len(outside_fov)} reference/axial-series combination(s) had their best candidate outside the image FOV.")
    else:
        report_lines.append("- No failure cases: all computed best-candidate projections landed inside the axial FOV.")
report_lines.append("")
report_lines.append("## Known limitations")
report_lines.append("")
for item in limitations + warnings:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## What Notebook 66 proves")
report_lines.append("")
report_lines.append(
    "> Pixel/patient coordinate transforms are mathematically correct (deterministic unit tests "
    "pass) and within-series DICOM geometry (orientation validity, physical slice ordering) is "
    "validated for real Sagittal T1, Sagittal T2 and Axial T2 series of a single lumbar study. "
    "Series discovery correctly classified all three roles from real DICOM metadata."
)
report_lines.append("")
report_lines.append("## What Notebook 66 DOES NOT prove")
report_lines.append("")
for item in [
    "no clinical validation",
    "no automatic pathology diagnosis",
    "no validated automatic disc naming",
    "no proof of registration under severe patient motion",
    "no proof across arbitrary scanners/protocols (validated on a single real study here)",
    "no guarantee when DICOM geometry is incomplete/inconsistent",
    "no deformable registration",
    "geometry_confidence is not a probability",
    "direct cross-series spatial correspondence is NOT validated for this study: "
    "FrameOfReferenceUID differs across all 3 series and no DICOM registration object was found "
    "(GATE E = PARTIAL, not PASS) -- raw cross-series distances are exploratory only",
]:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Recommended next notebook")
report_lines.append("")
report_lines.append(
    "Not decided automatically. The next technical requirement is "
    "`CROSS_FRAME_REGISTRATION`: resolving whether cross-series geometry should be validated as "
    "an extension of this notebook (e.g. `66b_postE50_cross_frame_registration.ipynb`) or as a "
    "precondition before `67_postE50_level_localization_v2.ipynb` attempts level/disc "
    "localization on top of unregistered series. This decision is deferred to the user."
)
report_lines.append("")

report_text = "\n".join(report_lines)
safe_write_text(REPORT_DIR / "post_e50_multiseries_geometry_pairing_report.md", report_text)
print(f"Report written: {(REPORT_DIR / 'post_e50_multiseries_geometry_pairing_report.md').relative_to(REPO_ROOT)}")
print(f"Report length: {len(report_text)} characters")


Report written: reports\post_e50\post_e50_multiseries_geometry_pairing_report.md
Report length: 14721 characters


## 19. EXPERIMENT STATUS


In [31]:
def found_or_not(role: str) -> str:
    if not len(series_inventory):
        return "NOT_FOUND"
    return "FOUND" if (series_inventory["candidate_role"] == role).any() else "NOT_FOUND"


WITHIN_SERIES_GEOMETRY_PASS = bool(gate_status["GATE_B_orientation"] == "PASS" and gate_status["GATE_D_slice_ordering"] == "PASS")
SAGITTAL_NUMERIC_COMPATIBILITY_PASS = bool(
    len(sagittal_pairing_metrics) > 0 and sagittal_pairing_metrics["geometry_numerically_compatible"].all()
) if len(sagittal_pairing_metrics) else False
SAME_FRAME_OF_REFERENCE_ANY = bool(len(series_inventory) and series_inventory["frame_of_reference_match"].any())
CROSS_FRAME_TRANSFORM_AVAILABLE = bool(len(registration_objects_found) > 0)
CROSS_SERIES_SPATIAL_RELATIONSHIP = gate_status["GATE_E_cross_series_spatial_relationship"]

if QUALITY_GATE_OVERALL == "PASS":
    decision = "GEOMETRY_PAIRING_VALIDATED_FOR_TEST_STUDY"
elif QUALITY_GATE_OVERALL == "PARTIAL":
    decision = "PARTIAL"
else:
    decision = "BLOCKED"

next_technical_requirement = "NONE" if QUALITY_GATE_OVERALL == "PASS" else "CROSS_FRAME_REGISTRATION"

recommended_next_notebook_text = (
    "67_postE50_level_localization_v2.ipynb"
    if QUALITY_GATE_OVERALL == "PASS"
    else "NOT_AUTO_DECIDED -- deferred to the user: resolve CROSS_FRAME_REGISTRATION either as an "
         "extension of Notebook 66 or as a precondition before Notebook 67 attempts level localization."
)

status_block = f'''EXPERIMENT STATUS

Experiment:
Post-E50 Multiseries DICOM Geometry Pairing

Training performed:
NO

Model modification:
NO

Checkpoint modification:
NO

DICOM copied into repository:
NO

Frozen E50 source modified:
NO

Git branch:
{GIT_BRANCH}

Git commit:
{GIT_COMMIT}

Notebook:
66_postE50_multiseries_geometry_pairing.ipynb

Real DICOM used:
{"YES" if REAL_DICOM_USED else "NO"}

Study opaque ID:
{STUDY_OPAQUE_ID if REAL_DICOM_USED else "N/A"}

Geometry unit tests:
{"PASS" if GEOMETRY_UNIT_TESTS_PASS else "FAIL"}

Series discovery:
{"PASS" if len(series_inventory) > 0 else "FAIL"}

Within-series geometry:
{"PASS" if WITHIN_SERIES_GEOMETRY_PASS else "FAIL"}

Sagittal T1:
{found_or_not("sagittal_t1")}

Sagittal T2:
{found_or_not("sagittal_t2")}

Axial T2:
{found_or_not("axial_t2")}

Coordinate transform:
{gate_status["GATE_C_transform_roundtrip"]}

Sagittal numeric compatibility:
{"PASS" if SAGITTAL_NUMERIC_COMPATIBILITY_PASS else "FAIL"}

Same Frame of Reference:
{"YES" if SAME_FRAME_OF_REFERENCE_ANY else "NO"}

Cross-frame transform available:
{"YES" if CROSS_FRAME_TRANSFORM_AVAILABLE else "NO"}

Cross-series spatial relationship:
{CROSS_SERIES_SPATIAL_RELATIONSHIP}

Axial orientation clustering:
{"PASS" if AXIAL_ORIENTATION_CLUSTERING_PASS else "FAIL"}
(clusters found: {len(axial_orientation_clusters)})

Privacy audit:
{gate_status["GATE_F_privacy"]}

Quality gate:
{QUALITY_GATE_OVERALL}

Decision:
{decision}

Next technical requirement:
{next_technical_requirement}

Recommended next notebook:
{recommended_next_notebook_text}

Warnings:
{chr(10).join(warnings) if warnings else "(none)"}
'''

print(status_block)


EXPERIMENT STATUS

Experiment:
Post-E50 Multiseries DICOM Geometry Pairing

Training performed:
NO

Model modification:
NO

Checkpoint modification:
NO

DICOM copied into repository:
NO

Frozen E50 source modified:
NO

Git branch:
research/post-e50-series-pairing

Git commit:
835373adf763e02be144e7086653a0a173f71879

Notebook:
66_postE50_multiseries_geometry_pairing.ipynb

Real DICOM used:
YES

Study opaque ID:
206aa67ee4e6

Geometry unit tests:
PASS

Series discovery:
PASS

Within-series geometry:
PASS

Sagittal T1:
FOUND

Sagittal T2:
FOUND

Axial T2:
FOUND

Coordinate transform:
PASS

Sagittal numeric compatibility:
PASS

Same Frame of Reference:
NO

Cross-frame transform available:
NO

Cross-series spatial relationship:
PARTIAL

Axial orientation clustering:
PASS
(clusters found: 5)

Privacy audit:
PASS

Quality gate:
PARTIAL

Decision:
PARTIAL

Next technical requirement:
CROSS_FRAME_REGISTRATION

Recommended next notebook:
NOT_AUTO_DECIDED -- deferred to the user: resolve CROSS_F